# Try Llama Storm for few-shot simplification

In [4]:
import pathlib as pth

base_location  = pth.Path.cwd().parent.parent

llama_location  = base_location / "models" / "llama_storm" / "Llama-3.1-Storm-8B.Q4_K_M.gguf"

## Load Model

I load model through LlamaCpp so it is much lighter

In [5]:
from langchain_community.llms import LlamaCpp

In [6]:
model = LlamaCpp(
            model_path=llama_location.as_posix(),
            n_gpu_layers=16,
            n_batch=512,
            temperature=0.8,
            max_tokens=256,
            top_p=5,
            verbose=False,
            n_ctx=8192,
            f16_kv=True,
            repeat_penalty=1.1,
        )

I set the temperature lower because we want the model to follow our instructions strictly.

## Let's try the zero-shot simplification tool by levels

In [7]:
def format_prompt(user_text: str, level: int) -> str:
    if level == 1:
        few_shot_examples = (
            """<|start_header_id|>user<|end_header_id|>\n\n"""
            "Simplify the following text in Russian: "
            "Первый будет игровым переложением мультсериала \"Бэтмен по ту сторону\", который дебютировал в прошлом году на WB Kids Network.\n"
            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
            "Первый фильм будет игровым переложением мультсериала «Бэтмен по ту сторону». Мультсериал начал выходить в прошлом году.\n"
            "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
            "Simplify the following text in Russian: "
            "Фильм будет ставить режиссер Боаз Якин (\"Свежий\").\n"
            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
            "Режиссёр Боаз Якин («Свежий») будет ставить фильм.\n"
            "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
            "Simplify the following text in Russian: "
            "Он уточнил, что на данный момент уже заключено семь государственных контрактов на выполнение ремонтных работ дорог в районах области, их стоимость оценивается в 3,2 миллиарда рублей.\n"
            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
            "Сейчас уже заключили семь контрактов на выполнение работ в области. Это стоит 3,2 миллиарда рублей.\n"
            "<|eot_id|>"
        )

        prompt = (
            """<|start_header_id|>system<|end_header_id|>\n\n"""
            "You are a text simplification tool for the Russian language. "
            "You are given a text in Russian and you need give a simple version of it in Russian. "
            "Your task is to simplify complex sentences into simple ones. "
            "Avoid cramming too many details into one sentence; distribute them across multiple sentences where needed. "
            "Rephrase sentences to remove participial and gerundial constructions. "
            "Where possible, replace passive voice with active voice. "
            "If a sentence consists of only a noun, add a verb. "
            "Replace rare or low-frequency words with more common ones. "
            "Where appropriate, remove or replace foreign words. "
            "Clarify ambiguous phrases by replacing them with more concrete, easily understandable words. "
            "Where possible, avoid words that have paronyms. "
            "Use only Russian, English is forbidden at any cost."
            "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
            + few_shot_examples +
            "Simplify the following text in Russian: {user_text}\n"
            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
        )

    elif level == 2:
        few_shot_examples = (
            """<|start_header_id|>user<|end_header_id|>\n\n"""
            "Simplify the following text in Russian: "
            "Первый будет игровым переложением мультсериала \"Бэтмен по ту сторону\", который дебютировал в прошлом году на WB Kids Network.\n"
            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
            "Первый фильм будет версией мультфильма «Бэтмен по ту сторону». Мультфильм вышел в прошлом году.\n"
            "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
            "Simplify the following text in Russian: "
            "Фильм будет ставить режиссер Боаз Якин (\"Свежий\").\n"
            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
            "Режиссёр Боаз Якин будет ставить фильм. Он известен по фильму «Свежий».\n"
            "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
            "Simplify the following text in Russian: "
            "Он уточнил, что на данный момент уже заключено семь государственных контрактов на выполнение ремонтных работ дорог в районах области, их стоимость оценивается в 3,2 миллиарда рублей.\n"
            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
            "Заключили семь контрактов на выполнение работ. Это стоит 3,2 миллиарда рублей.\n"
            "<|eot_id|>"
        )

        prompt = (
            """<|start_header_id|>system<|end_header_id|>\n\n"""
            "You are a text simplification tool for the Russian language. "
            "You are given a text in Russian and you need give a simple version of it in Russian. "
            "Simplify complex or compound sentences by breaking them into shorter ones, aiming for a sentence length of no more than seven words. "
            "Ensure each sentence contains only one idea. "
            "Avoid participial and gerundial constructions, and prefer active voice over passive voice. "
            "Keep essential information like names, nationalities, and roles. Do not remove important details. "
            "Remove unnecessary foreign words (like brand names) and replace rare or long words with simpler, shorter ones. "
            "Simplify ambiguous phrases by using more concrete, clear words. "
            "Remove minor details that do not add significant meaning, but ensure key information remains intact. "
            "Use only Russian, English is forbidden at any cost."
            "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
            + few_shot_examples +
            "Simplify the following text in Russian: {user_text}\n"
            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
        )

    elif level == 3:
        few_shot_examples = (
            """<|start_header_id|>user<|end_header_id|>\n\n"""
            "Simplify the following text in Russian: "
            "Первый будет игровым переложением мультсериала \"Бэтмен по ту сторону\", который дебютировал в прошлом году на WB Kids Network.\n"
            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
            "Первый фильм будет снять по мультфильму «Бэтмен по ту сторону». Мультфильм вышел в прошлом году.\n"
            "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
            "Simplify the following text in Russian: "
            "Фильм будет ставить режиссер Боаз Якин (\"Свежий\").\n"
            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
            "Фильм снимет режиссёр Боаз Якин. Он известен по фильму «Свежий».\n"
            "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
            "Simplify the following text in Russian: "
            "Он уточнил, что на данный момент уже заключено семь государственных контрактов на выполнение ремонтных работ дорог в районах области, их стоимость оценивается в 3,2 миллиарда рублей.\n"
            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
            "Уже заключили договоры на ремонт. Он стоит 3.2 миллиарда рублей.\n"
            "<|eot_id|>"
        )

        prompt = (
            """<|start_header_id|>system<|end_header_id|>\n\n"""
            "You are a text simplification assistant for the Russian language. "
            "You are given a text in Russian and you need give a simple version of it in Russian. "
            "Your task is to make the text as simple as possible. "
            "Each sentence should contain only one idea and be no longer than five words. "
            "Remove or replace foreign words (such as names, places, or brands), and avoid minor details. "
            "Eliminate numbers and remove any unnecessary details. "
            "Focus on using the nominative and genitive cases for nouns, and only the present or past tense for verbs. "
            "Avoid passive voice and inverted word order. "
            "Replace rare or low-frequency words with more common ones. "
            "Where possible, replace complex phrases with common expressions, clichés, or idioms. "
            "Remove any extraneous details (if it is possible to without removing original semantics of sentence) and simplify ambiguous phrases as much as possible."
            "Use only Russian, English is forbidden at any cost."
            "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
            + few_shot_examples +
            "Simplify the following text in Russian: {user_text}\n"
            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
        )

    return prompt.format(user_text=user_text)


### 1 Level

In [8]:
prompt = format_prompt(user_text="Россиянка Елена Максимова одержала победу в международном конкурсе «Миссис Вселенная».", level=1)

In [9]:
model.invoke(str(prompt))

'Россиянка Елена Максимова стала победительницей конкурса «Миссис Вселенная».'

### 2 Level

In [10]:
prompt = format_prompt(user_text="Россиянка Елена Максимова одержала победу в международном конкурсе «Миссис Вселенная».", level=2)

In [11]:
model.invoke(str(prompt))

'Россиянка Елена Максимова стала победительницей конкурса «Миссис Вселенная».'

### 3 Level

In [12]:
prompt = format_prompt(user_text="Россиянка Елена Максимова одержала победу в международном конкурсе «Миссис Вселенная».", level=3)

In [13]:
model.invoke(str(prompt))

'Россиянка Елена Максимова выиграла конкурс «Миссис Вселенная».'

Well, not bad at all! Let's compute the 200 samples from dataset.

## Load Data


In [14]:
import pandas as pd

data_location = base_location / "data" / "RuSimpleSentAphasia.csv"
data = pd.read_csv(data_location.as_posix())

data.head()

,source,level 1,level 2,level 3
0,Россиянка Елена Максимова одержала победу в ме...,Россиянка Елена Максимова победила в конкурсе ...,Россиянка победила в конкурсе «Миссис Вселенная».,Россиянка победила в конкурсе «Миссис Вселенная».
1,Представительница России впервые завоевала это...,"В прессе сказали, что участница из России полу...",Участница из России получает этот титул впервые.,Россиянка получает этот титул впервые.
2,"Уточняется, что финал прошел в Софии 4 февраля.",Финал прошел в Болгарии в начале февраля.,Финал был в Болгарии в феврале.,Финал был начале февраля. Он был в в Болгарии.
3,Участие в нем принимали 120 женщин из разных с...,В нем участвовали 120 женщин из разных стран.,В нем участвовали 120 женщин из разных стран.,В нем участвовали женщины из разных стран.
4,«Конкуренция на конкурсе была очень жесткая: р...,«В конкурсе было сложно выиграть. Было много у...,«В конкурсе было сложно выиграть. Было много у...,«В конкурсе было сложно выиграть.


In [15]:
len(data)

1002

In [16]:
new_data = pd.DataFrame(data["source"].head(200))

In [17]:
new_data

,source
0,Россиянка Елена Максимова одержала победу в ме...
1,Представительница России впервые завоевала это...
2,"Уточняется, что финал прошел в Софии 4 февраля."
3,Участие в нем принимали 120 женщин из разных с...
4,«Конкуренция на конкурсе была очень жесткая: р...
...,...
195,Ранее президент Национального института геофиз...
196,Во время операции врачи извлекли 39 металличес...
197,"По словам одного из хирургов, этот пациент поп..."
198,"Выяснилось, что все эти предметы пациент прогл..."


In [18]:
def apply_generation(text, level) -> str:
    prompt = format_prompt(user_text=text, level=level)
    return model.invoke(str(prompt), stop=["\n\n", ". ", " \n\n", ". \n\n", "\n", " \n"])

In [19]:
for level in [1, 2, 3]:
    column = f"level {level}"
    new_data[column] = new_data["source"].apply(lambda x: apply_generation(x, level))

In [20]:
new_data_location = base_location / "data" / "RuSimpleSentAphasia_200_generated_llama_few_shot.csv"

new_data.to_csv(new_data_location.as_posix(), index=False)

## Let's calculate BERTscore between ground truth and predicted texts

In [21]:
new_data

,source,level 1,level 2,level 3
0,Россиянка Елена Максимова одержала победу в ме...,Россиянка Елена Максимова стала победительнице...,Россиянка Елена Максимова стала победительнице...,Россиянка Елена Максимова выиграла конкурс «Ми...
1,Представительница России впервые завоевала это...,Российская участница впервые получила этот титул.,Российская девушка впервые стала победительницей,Россиянка выиграла титул
2,"Уточняется, что финал прошел в Софии 4 февраля.",Финал прошел 4 февраля в Софии.,Финал прошел 4 февраля в Софии.,Финал прошел 4 февраля в Софии.
3,Участие в нем принимали 120 женщин из разных с...,В этом проекте участвовали 120 женщин из разны...,В мероприятии участвовали 120 женщин из разных...,В акции участвовали 120 женщин из разных стран.
4,«Конкуренция на конкурсе была очень жесткая: р...,Конкурс был очень конкурентным: рекордное коли...,Конкурс был очень сложным,Конкурс был очень сложным
...,...,...,...,...
195,Ранее президент Национального института геофиз...,Президент Италии по геофизике и вулканологии К...,В Турции произошло сильное землетрясение,Землетрясение сдвинуло землю на три метра.
196,Во время операции врачи извлекли 39 металличес...,Во время операции у 30-летнего ливанца извлекл...,Врачи извлекли 39 металлических и пластмассовы...,Врачи извлекли 39 ножей и других предметов из ...
197,"По словам одного из хирургов, этот пациент поп...",Пациент попал в больницу с приступом удушья,Пациент попал в больницу с приступом удушья,Пациент попал в больницу с удушьем
198,"Выяснилось, что все эти предметы пациент прогл...","Выяснилось, что за год пациент проглотил все э...",В течение года пациент проглотил все эти предм...,Пациент проглотил все это за год.


In [22]:
from evaluate import load
import numpy as np

bertscore = load("bertscore")

bert_scores = {}
val_data = data.head(200)


for level in val_data.columns[1:]:
    references = val_data[level].tolist()
    predictions = new_data[level].tolist()

    results = bertscore.compute(predictions=predictions, references=references, lang="ru")
    
    bert_scores[level] = {
        "precision": np.mean(results['precision']),
        "recall": np.mean(results['recall']),
        "f1": np.mean(results['f1'])
    }

/home/z00logist/hse-simplification-for-aphasia/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/z00logist/hse-simplification-for-aphasia/.venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [23]:
for level, scores in bert_scores.items():
    print(f"BERTScore for {level}:")
    print(f"Precision: {scores['precision']}")
    print(f"Recall: {scores['recall']}")
    print(f"F1: {scores['f1']}\n")

BERTScore for level 1:
Precision: 0.8124954956769943
Recall: 0.8140748885273933
F1: 0.8123864933848381

BERTScore for level 2:
Precision: 0.7798861038684844
Recall: 0.7780787941813468
F1: 0.7779801541566849

BERTScore for level 3:
Precision: 0.7733590400218964
Recall: 0.7571183815598488
F1: 0.7643417346477509



Few shot made llama have better simplifications!